Working with Qwen3-vl-32B

In [1]:
def kv_cache_bytes(layers, kv_heads, head_dim, seq_len, batch, dtype_bytes=2):
    # the 2 = K and V, the two caches
    return 2 * layers * kv_heads * head_dim * seq_len * batch * dtype_bytes

def gib(b): return b / 1024**3

# --- Qwen3-VL-32B-Instruct, real text_config ---
cfg = dict(layers=64, kv_heads=8, head_dim=128, dtype_bytes=2)  # bf16

In [2]:
per_tok = kv_cache_bytes(**cfg, seq_len=1, batch=1)
print(per_tok, "bytes/token =", per_tok/1024, "KiB/token")

262144 bytes/token = 256.0 KiB/token


In [3]:
per_4k = kv_cache_bytes(**cfg, seq_len=4096, batch=1)
print(gib(per_4k), "GiB per 4k session")

1.0 GiB per 4k session


In [4]:
TOTAL_VRAM_GIB   = 128          # 4 × 32 GiB
UTIL             = 0.88         # your gpu_memory_utilization
WEIGHTS_GIB      = 62           # ~32B params in bf16 (+ small vision tower)

usable = TOTAL_VRAM_GIB * UTIL
kv_budget = usable - WEIGHTS_GIB
sessions = kv_budget / gib(per_4k)
print(f"usable {usable:.0f} GiB - weights {WEIGHTS_GIB} = {kv_budget:.0f} GiB KV")
print(f"-> ~{sessions:.0f} concurrent 4k sessions")

usable 113 GiB - weights 62 = 51 GiB KV
-> ~51 concurrent 4k sessions


In [9]:
mha = dict(cfg); mha["kv_heads"] = 64      # 8 -> 64
per_token_mha = kv_cache_bytes(**mha, seq_len=1, batch=1)
per_4k_mha = kv_cache_bytes(**mha, seq_len=4096, batch=1)

print(per_token_mha, "bytes/token =", per_token_mha/1024, "KiB/token " , per_token_mha/1024 ** 2, "MiB/token")
print(gib(per_4k_mha), "GiB per 4k session (MHA)")
print("ratio:", per_4k_mha / per_4k, "x")
print("MHA sessions:", (usable - WEIGHTS_GIB) / gib(per_4k_mha))

2097152 bytes/token = 2048.0 KiB/token  2.0 MiB/token
8.0 GiB per 4k session (MHA)
ratio: 8.0 x
MHA sessions: 6.33


The kv cache calculation and no of concurrent session calcucation is directly based on the, size per token , and no of tokens per request, to calculate the kv cache per token, we use the formula, 2 * layers * kv_heads * head_size  * dtype byte , so for the Qwen model we got 256Kib / token. If we assume 4k tokens we are getting around 1 gb kv cache per request, with that and if we assume we have 128gb vram with 0.88 gpu util, we get 113 usable memory where model weights themselves take up 62gb so we get 51 gb vram for kv cache marking for 51 concurrent sessions with GQA - Grouped Query Attention. where a group query heads will attend to a single kv head. Here the num_attention_heads are 64 and k_v heads are 8, making each kv head is attended by the group of 8 query heads. The same config when done with MHA - Multi Headed Attention, the classic one I guess, is each query head should attend to each kv_head making, kv heads = query heads, thus reducing number of sessions , and increasing the requirement of gpu space per token to 2048Kib per token with MHA from 256Kib per token with GQA. with our calculation we can also notice that the scale is 8X with MHA over GQA. effectively dropping the number of sessions 51 with GQA to around 6 session with MHA. So GQA is preferred.

KV cache per token = 2 × layers × kv_heads × head_dim × dtype_bytes (the 2 = K and V). Note: per-TOKEN excludes seq_len and batch; multiply by seq_len × batch to get per-REQUEST. For Qwen3-VL-32B (64 layers, 8 KV heads, head_dim 128, bf16): 256 KiB/token → at 4k context, exactly 1 GiB/request. With 128 GiB VRAM × 0.88 util = 113 GiB usable, minus ~62 GiB weights = 51 GiB for KV → ~51 concurrent 4k sessions.

This is GQA: 64 query heads share 8 KV heads (group size 8). MHA is the classic form where kv_heads = num_attention_heads — the ONLY variable that changes is kv_heads (8 → 64); layers, head_dim, dtype stay identical, which is why the ratio is exactly 64/8 = 8×. Under MHA the same model costs 2 MiB/token and the 51 GiB KV budget fits only ~6 concurrent sessions. So GQA buys 8× the concurrency for free by sharing KV heads — the core reason modern models use it.